In [1]:
import numpy as np
import copy
import config as cfg
import helpers
from data_acquirer import get_data, gen_data

In [2]:
method = 2
random_data = True
total_robots = 5
total_tasks = 50
data_name = 'data4'

In [168]:
class Path:
    def __init__(self,depot):
        self.tasks =[depot]
        self.total_time = 0
        self.travel_time = 0
        pass
    
    def insert(self, task):
        self.tasks.append(task)
        self.update_travel_time(task)
        return
    
    def update_total_time(self, time):
        self.total_time = time
        return
    
    def update_travel_time(self,task):
        last_task = self.get_last_task()
        self.travel_time += helpers.cal_dist_task(last_task, task)
#         return travel_time
    
    def get_insert_cost(self, task, c_travel, c_operate):
        last_task = self.get_last_task()
        cost_if_insert = self.total_time + c_travel[last_task.index][task.index] + c_operate[task.index]
        return cost_if_insert
    
    def get_last_task(self):
        return self.tasks[-1]
    
    def __repr__(self):
            return f"Path {[task.index for task in self.tasks]}"

In [169]:
class Solution:
    def __init__(self, paths, depot, tasks):
        self.paths = paths
        self.depot = depot
        self.tasks = tasks
        
    def __repr__(self):
        return f"Paths {self.paths}"

In [216]:
is_depot_task = True
if random_data:
    raw = gen_data(total_robots, total_tasks, False, is_depot_task)
else:
    raw = get_data(data_name,'csv', is_depot_task)

In [218]:
raw.tasks

[Task 1,[332.09108067  40.7080629 ],[442.19847527 158.81268462],1,
 Task 2,[143.76443274 195.8748978 ],[305.54088446 445.87541604],3,
 Task 3,[79.12095762 16.04974799],[449.55507232 344.87470332],4,
 Task 4,[ 88.16405655 431.20356408],[434.94757104 421.32392654],4,
 Task 5,[235.52917372 448.33085407],[413.12229105 189.94318254],2,
 Task 6,[ 74.88775547 283.88008678],[ 57.1902104  440.15885698],4,
 Task 7,[135.41577318  95.89207183],[382.90136047 230.78651811],1,
 Task 8,[205.48360751 474.78068791],[388.76772876 264.22868325],4,
 Task 9,[262.74631195 407.68597444],[465.63590076 205.33111252],5,
 Task 10,[243.08703605 220.54493908],[ 74.58603409 396.9116932 ],1,
 Task 11,[392.92142487 487.77016729],[ 92.17050635 313.80826861],1,
 Task 12,[475.29059194 159.26769813],[387.61939453 304.05376935],4,
 Task 13,[130.48541501 485.09496068],[152.01462855 423.90854027],3,
 Task 14,[170.79080591  86.60930794],[ 0.80119249 36.69190923],4,
 Task 15,[157.22413327 283.72634428],[240.67534817 303.259588

In [199]:
c_travel = helpers.create_task_dist_dict(raw.depot,raw.tasks)
c_operate = helpers.create_operate_dict(raw.depot,raw.tasks)
for i in range(len(c_travel)):
    c_travel[i].pop(i)


uninserted = copy.deepcopy(raw.tasks) #j left
paths = [Path(raw.depot) for _ in range(total_robots)]
cur_sol = Solution(paths,raw.depot, raw.tasks)
best_obj = 0

idx_task_mapper = {task.index: task for task in raw.tasks}

C_max = 0
while len(uninserted) > 0:
    shortest_path_length = 9_999_999
    for path in cur_sol.paths:
        last_task = path.get_last_task()
        last_path_idx = last_task.index
        closest_idx = min(c_travel[last_path_idx],key = c_travel[last_path_idx].get)
        closest_task = idx_task_mapper[closest_idx]
        cost_if_insert = path.get_insert_cost(closest_task,c_travel,c_operate)
#         delta  = cost_if_insert - C_max
    
        if cost_if_insert < shortest_path_length:
            shortest_path_length = cost_if_insert
            best_path = path
            best_task = idx_task_mapper[closest_idx]
    
    print(f"BEST PATH: {best_path} \n BEST TASK: {best_task}")
    costs_if_insert = {path: path.get_insert_cost(best_task,c_travel, c_operate)}
#     print(f'c_travel {c_travel}')
    sorted_cost = list(np.sort(list(costs_if_insert.values())))
    shortest_to_insert_costs = sorted_cost[:best_task.n]
    inserted_paths_length = shortest_to_insert_costs[-1]
    for _ in range(best_task.n):
        cur_length = shortest_to_insert_costs.pop()
        for path, length in costs_if_insert.items():
            if length == cur_length:
                path.insert(best_task)
                path.update_total_time(inserted_paths_length)
                
    for i in range(len(c_travel)):
        try:
            c_travel[i].pop(best_task.index)
        except:
            pass
    
    uninserted.pop(best_task)
    # Remember to remove best_task from c_travel

BEST PATH: Path [0] 
 BEST TASK: Task 39,[106.2817637  386.54541632],[ 18.05951054 499.99225707],5


IndexError: pop from empty list

In [212]:
some_list = [69,30]
some_list.remove(30)

In [215]:
c_travel

{0: {1: 205.38352145004262,
  2: 330.5746976357832,
  4: 331.73438131055724,
  5: 130.70201389018493,
  6: 462.8704755886657,
  7: 47.85753621443574,
  8: 78.32231292174166,
  9: 52.845607042614816,
  10: 325.36364876587385,
  11: 262.8153420847011,
  12: 57.57702143017476,
  13: 249.35772892950416,
  14: 284.33781087311513,
  15: 116.29907845615075,
  16: 61.67152891303426,
  17: 465.88003167169563,
  18: 198.01880061688348,
  19: 109.74602414035705,
  20: 193.62842549230788,
  21: 163.70810146587772,
  22: 346.3440490526702,
  23: 326.45634671616153,
  24: 215.90206256422377,
  25: 323.2796182333414,
  26: 93.07859848624383,
  27: 133.4058933576663,
  28: 112.96970997728421,
  29: 435.76092112922026,
  30: 59.81019313307948,
  31: 262.4646613622698,
  32: 313.75063612319644,
  33: 323.48226541547433,
  34: 50.70888624133648,
  35: 495.5499008544004,
  36: 431.46433961203803,
  37: 318.76514166617744,
  38: 286.5100539997024,
  39: 36.52352404945473,
  40: 98.04231675694426,
  41: 243

In [ ]:
# while len(uninserted) > 0:
#     cur_task = uninserted.pop()
#     deltas = {}
#     for path in paths:
#         cur_time = path.cost
#         travel_time = helpers.cal_dist_task(path.last_task(),cur_task)
#         process_time = cur_task.process_time
#         cost_if_insert = cur_time + travel_time + process_time
#         delta = cost_if_insert - best_obj
#         deltas[path] = delta

In [ ]:
def insert_single(solution, task):
    """
    Find a best spot to insert the `task` into `solution`
    
    Parameters
    ----------
    solution: Solution
    current solution
    task : Task
    current task want to insert to the solution, note that this must be a single task.
    """
    for path in soltion.paths: 
        for i in range(len(path)):
            temp_path = path.insert(i,task)
            

In [ ]:
def evaluate_after_insert(new_path,old_path):
    

In [ ]:
abc = [1,2,3,4]
abc.insert(0,69)